## Cell 1 — Paths and environment

Locates `bank_pipeline/` by walking up from the working directory, makes it the working directory and the import root, so the notebook runs from anywhere in the tree and every relative artifact path (`./models/...`) points at the folders the current producers write.

`BLACKBOX_ROOT` holds this arm's Stage 3-4 artifact folders.

Because `data` becomes a top-level import, Cell 2 verifies it resolves to `bank_pipeline/data`.


In [ ]:
# ============================================================
# CELL 1: CONFIGURE PATHS AND ENVIRONMENT
# ============================================================

import os, sys
from pathlib import Path
import numpy as np

BANK_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "bank_pipeline"), None
)
if BANK_ROOT is None:
    raise RuntimeError(f"bank_pipeline/ not found above {Path.cwd()}")
os.chdir(BANK_ROOT)
if str(BANK_ROOT) not in sys.path:
    sys.path.insert(0, str(BANK_ROOT))

print(f"📁 import root : {BANK_ROOT}")
print(f"📂 working dir : {Path.cwd()}")
for pkg in ("data", "shared", "glass_pipeline", "blackbox_pipeline"):
    print(f"   {'✅' if (BANK_ROOT / pkg).is_dir() else '❌ MISSING'} {pkg}/")

# This arm's Stage 3-4 artifact folders live under blackbox_pipeline/models
BLACKBOX_ROOT  = BANK_ROOT / "blackbox_pipeline"

## Cell 2 — Shared GLASS database

Opens the SQLite database shared by both arms. `build_database()` does nothing if `raw_bank` already exists and is never called with `rebuild=True`, so shared data cannot be overwritten.

The assertion verifies `bank_database` came from `BANK_ROOT`, guarding against module shadowing after Cell 1 modifies the path. If it fails, restart the kernel.

In [ ]:
# ============================================================
# CELL 2: INITIALIZE SHARED BANK DATABASE
# ============================================================
from data import bank_database
from data.bank_database import DEFAULT_DB_PATH, RAW_TABLE, build_database, raw_bank_exists

# Make sure `data` resolved to bank_pipeline/data, not another pipeline's package
assert BANK_ROOT in Path(bank_database.__file__).resolve().parents, (
    f"`data` imported from {bank_database.__file__}; restart the kernel"
)

existed_before = raw_bank_exists(DEFAULT_DB_PATH)
DB_PATH = build_database()  # no-op when raw_bank exists; never passes rebuild=True

print(f"database: {DB_PATH}  ({'existing' if existed_before else 'built from CSV'})")

## Cell 3 — Load and validate raw data

Reads `raw_bank` through a **read-only** connection (`mode=ro`). `ORDER BY rowid` preserves deterministic row order for the downstream split.

Assertions pin the dataset to `bank-additional-full.csv`: 41,188 rows, 21 columns, and `y ∈ {yes, no}`.

In [ ]:
# ============================================================
# CELL 3: LOAD AND VALIDATE RAW BANK DATA
# ============================================================
import sqlite3
from contextlib import closing

import pandas as pd

EXPECTED_ROWS    = 41_188  # bank-additional-full.csv
EXPECTED_N_COLS  = 21

# Read-only connection: this notebook must never modify the shared database
with closing(sqlite3.connect(DB_PATH.as_uri() + "?mode=ro", uri=True)) as conn:
    df_raw = pd.read_sql_query(f"SELECT * FROM {RAW_TABLE} ORDER BY rowid", conn)

assert len(df_raw) == EXPECTED_ROWS, f"Unexpected row count: {len(df_raw):,}"
assert df_raw.shape[1] == EXPECTED_N_COLS, f"Unexpected column count: {df_raw.shape[1]}"
assert df_raw["y"].isin(["yes", "no"]).all(), "Unexpected target values in raw_bank"

print(f"raw_bank: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(pd.DataFrame({
    "count": df_raw["y"].value_counts(),
    "share": df_raw["y"].value_counts(normalize=True).round(4),
}).to_string())

## Cell 4 — GLASS imports

Imports the GLASS components Stage 1 depends on: `BankPreprocessor`, `GlobalSplitManager`, `engineer_features`, and `LR_FEATURES`.

`quiet()` suppresses GLASS output unless `SHOW_GLASS_LOGS = True`. `LR_FEATURES` is copied to a list for a stable local sequence.

In [ ]:
# ============================================================
# CELL 4: IMPORT NECESSARY GLASS MODULES
# ============================================================

import io
from contextlib import redirect_stdout

from data.preprocessing.bank_preprocessing import BankPreprocessor
from data.global_splits.split_manager import GlobalSplitManager
from shared.stage1.feature_engineering import LR_FEATURES, engineer_features

LR_FEATURES = list(LR_FEATURES)

# GLASS modules print verbose logs; set True to see them
SHOW_GLASS_LOGS = False

def quiet(fn, *args, **kwargs):
    """Call a GLASS function, hiding its stdout unless SHOW_GLASS_LOGS is True."""
    if SHOW_GLASS_LOGS:
        return fn(*args, **kwargs)
    with redirect_stdout(io.StringIO()):
        return fn(*args, **kwargs)

## Cell 5 — Verify module sources

Prints each GLASS module source and the Stage 1 feature list.

This provenance check catches stale or shadowed imports before modelling. The displayed `LR_FEATURES` is the same contract validated by `blackbox_pipeline.models.mlp.features.assert_matches_glass`.

In [ ]:
# ============================================================
# CELL 5: VERIFY GLASS MODULE SOURCES AND STAGE 1 FEATURES
# ============================================================

import sys
for obj in (BankPreprocessor, GlobalSplitManager, engineer_features):
    print(f"{obj.__name__:<20} ← {sys.modules[obj.__module__].__file__}")
print("LR_FEATURES:", LR_FEATURES)

## Cell 6 — Global split

`BankPreprocessor(drop_leaky=True)` removes leakage features, then `GlobalSplitManager` creates the shared 80/20 split with `random_state=42`.

Both GLASS LR and the MLP use identical train/test rows, isolating model differences from data differences.

The assertion verifies no leaky columns remain in `feature_names`.

In [ ]:
# ============================================================
# CELL 6: BUILD GLOBAL SPLIT
# ============================================================

RANDOM_STATE = 42

preprocessor = BankPreprocessor(drop_leaky=True, verbose=False)
df_processed = preprocessor.fit_transform()

split_manager = GlobalSplitManager(test_size=0.20, random_state=RANDOM_STATE)
GLOBAL_SPLIT  = quiet(split_manager.create_split, df_processed, target_col="y")

assert not set(BankPreprocessor.LEAKY_FEATURES) & set(GLOBAL_SPLIT["feature_names"])
print(f"GLOBAL_SPLIT: train {GLOBAL_SPLIT['X_train'].shape} | test {GLOBAL_SPLIT['X_test'].shape}")

## Cell 7 — Stage 1 data

`engineer_features` is fit on training data and applied to both splits. It receives `y_train` but never test labels, preventing target leakage into engineered features.

`STAGE1` stores the arrays used downstream. The assertion fixes the Stage 1 feature contract (2.0.0) to `cellular_downturn`, `subscribe_rate_by_euribor_bin`, and `subscribe_rate_by_dow_month` (renamed in PR 39 from `cellular_crisis`, `euribor3m_local_rate`, and `dow_month_encoded`; values unchanged).


In [ ]:
# ============================================================
# CELL 7: BUILD STAGE 1 DATA
# ============================================================

X_train_s1, X_test_s1, stage1_engineer = quiet(
    engineer_features,
    GLOBAL_SPLIT["X_train"], GLOBAL_SPLIT["X_test"], GLOBAL_SPLIT["y_train"],
)

STAGE1 = {
    "X_train": X_train_s1,
    "X_test":  X_test_s1,
    "y_train": GLOBAL_SPLIT["y_train"],
    "y_test":  GLOBAL_SPLIT["y_test"],
}

assert set(X_train_s1.columns) == {"cellular_downturn", "subscribe_rate_by_euribor_bin", "subscribe_rate_by_dow_month"}
print(f"STAGE1: train {X_train_s1.shape} | test {X_test_s1.shape}")
print(STAGE1["y_train"].value_counts(normalize=True).round(4).to_dict())

## Cell 8 — Stage 1 imports and parameters

Imports the Stage 1 package API and sets the tuning budget.

`assert_matches_glass(LR_FEATURES)` verifies the MLP uses the same features as GLASS LR and reports both lists on mismatch.

`STAGE1_N_TRIALS = 100` matches GLASS's tuning budget. The printed module path confirms which package copy was imported.

In [ ]:
# ============================================================
# CELL 8: SET STAGE 1 MLP PARAMETERS
# ============================================================

from blackbox_pipeline.models.mlp import (
    CalibratedStage1MLP,
    assert_matches_glass,
)

assert_matches_glass(LR_FEATURES)

STAGE1_N_TRIALS = 100  # same as GLASS

print(f"mlp ← {sys.modules[CalibratedStage1MLP.__module__].__file__}")

## Cell 9 — Fit and tune

Runs the training-only Stage 1 pipeline: scaling → 100-trial Optuna search on 10-fold CV ROC-AUC → final fit → calibration → OOF F2 threshold selection.

`trials` reports completed versus pruned trials. `calibration` shows the shared Stage 1 family, `"sigmoid"` (PR 39, Step 2A). `CV threshold` comes from the GLASS-matched 0.05–0.49 grid in 0.01 steps.

This is the most expensive Stage 1 cell because each trial uses 10-fold CV.


In [ ]:
# ============================================================
# CELL 9: FIT AND TUNE STAGE 1 MLP
# ============================================================

stage1 = CalibratedStage1MLP(
    calibration_method="sigmoid",  # the shared Stage 1 family (PR 39, Step 2A)
    cv_folds=10,
    n_trials=STAGE1_N_TRIALS,
    random_state=RANDOM_STATE,
)
stage1.fit(
    STAGE1["X_train"], STAGE1["y_train"],
    X_train_raw=GLOBAL_SPLIT["X_train"],  # target encodings refit inside each CV fold (PR 36, F3)
)

_states = stage1.tuning_trials["state"].value_counts().to_dict()
print(f"trials: {_states}")
print(f"best CV ROC-AUC: {stage1.best_cv_roc_auc:.4f}")
print(f"calibration: {stage1.calibration_method}")
print(f"CV threshold: {stage1.optimal_threshold:.2f}  (CV F2 {stage1.cv_f2:.4f})")

## Cell 10 — Package outputs

Stores downstream results in `STAGE1_OUTPUT`.

`proba_train` is in-sample and diagnostic only. `proba_train_oof` is out-of-fold, so each row was scored by a model that did not train on it and is the valid series for tuning and reporting.

Assertions verify probability arrays remain aligned with their corresponding splits.

In [ ]:
# ============================================================
# CELL 10: PACKAGE STAGE 1 OUTPUTS
# ============================================================

STAGE1_OUTPUT = {
    "model":            stage1,
    "threshold":        stage1.optimal_threshold,
    "threshold_source": stage1.THRESHOLD_SOURCE,
    "proba_train_oof":  stage1.proba_train_oof,
    "proba_test":       stage1.predict_proba(STAGE1["X_test"]),
    "fold_id":          stage1.train_fold_id,
}

STAGE1_OUTPUT["pred_train_oof"] = (STAGE1_OUTPUT["proba_train_oof"] >= stage1.optimal_threshold).astype("int8")
STAGE1_OUTPUT["pred_test"]      = (STAGE1_OUTPUT["proba_test"] >= stage1.optimal_threshold).astype("int8")

STAGE1_PROBA_TRAIN_INSAMPLE = stage1.predict_proba(STAGE1["X_train"])   # diagnostic only

assert STAGE1_OUTPUT["proba_train_oof"].index.equals(STAGE1["X_train"].index)
assert STAGE1_OUTPUT["fold_id"].index.equals(STAGE1["X_train"].index)
assert STAGE1_OUTPUT["proba_test"].index.equals(STAGE1["X_test"].index)

print(pd.Series(stage1.describe()).to_string())

## Cell 11 — Evaluate

Reports metrics at 0.50 and the CV-selected threshold for train OOF and held-out test data.

Train OOF reflects the data used to choose the threshold; test provides the held-out estimate.

`oof_provenance` must be `refittable`. A `PREFIT` result would make the OOF scores effectively in-sample and invalidate threshold tuning.

In [ ]:
# ============================================================
# CELL 11: EVALUATE STAGE 1 PERFORMANCE
# ============================================================

stage1_metrics = pd.concat([
    stage1.metrics_table(STAGE1["y_train"], STAGE1_OUTPUT["proba_train_oof"], "train OOF"),
    stage1.metrics_table(STAGE1["y_test"],  STAGE1_OUTPUT["proba_test"],      "test"),
])

print(
    f"Stage 1 calibrated MLP ({stage1.calibration_method}) — "
    f"metrics @ 0.50 and @ CV threshold {stage1.optimal_threshold:.2f}"
)
print(f"OOF provenance: {stage1.oof_provenance_}")
print(stage1_metrics.round(4).to_string())

## Cell 12 — Plot setup

Prepares the threshold, probability series, and metric rows used by Cell 13. Produces no output.

> **Run order matters.** Cell 12 expects Cell 11's labels. Cell 14 rebuilds `stage1_metrics` with different labels, so re-run Cell 11 before Cell 12 if Cell 14 has already run.

In [ ]:
# ============================================================
# CELL 12: PLOT STAGE 1 DISCRIMINATION AND CALIBRATION
# ============================================================

import matplotlib.pyplot as plt

thr = stage1.optimal_threshold
y_te, p_te = STAGE1["y_test"], STAGE1_OUTPUT["proba_test"]
y_tr, p_oof = STAGE1["y_train"], STAGE1_OUTPUT["proba_train_oof"]

m_te_05  = stage1_metrics.loc["test @ 0.50"]
m_te_thr = stage1_metrics.loc[f"test @ {thr:.2f}"]
m_tr_05  = stage1_metrics.loc["train OOF @ 0.50"]


## Cell 13 — Diagnostic panel

Plots test and train-OOF diagnostics for discrimination, calibration, and probability separation.

ROC and PR show ranking quality; PR is especially informative at the ~11% positive rate. Reliability plots compare predicted and observed probabilities, with Brier and ECE summarising calibration.

Reliability uses quantile bins so each bin contains similar sample counts.

In [ ]:
# ============================================================
# CELL 13: PLOT STAGE 1 THRESHOLD AND PROBABILITY DIAGNOSTICS
# ============================================================

from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve, roc_curve

fig, axes = plt.subplots(2, 3, figsize=(16, 9.5))
fig.suptitle(f"Stage 1 calibrated MLP ({stage1.calibration_method}) — diagnostics",
             fontsize=13, fontweight="bold")

# ROC
ax = axes[0, 0]
for label, yy, pp, ls, m in (("test", y_te, p_te, "-", m_te_05), ("train OOF", y_tr, p_oof, "--", m_tr_05)):
    fpr, tpr, _ = roc_curve(yy, pp)
    ax.plot(fpr, tpr, ls, label=f"{label}  AUC {m['roc_auc']:.3f}")
ax.plot([0, 1], [0, 1], ":", color="grey")
ax.set(title="ROC curve", xlabel="False positive rate", ylabel="True positive rate")
ax.legend(loc="lower right")

# Precision-recall
ax = axes[0, 1]
for label, yy, pp, ls, m in (("test", y_te, p_te, "-", m_te_05), ("train OOF", y_tr, p_oof, "--", m_tr_05)):
    prec, rec, _ = precision_recall_curve(yy, pp)
    ax.plot(rec, prec, ls, label=f"{label}  AP {m['pr_auc']:.3f}")
ax.axhline(y_te.mean(), ls=":", color="grey", label=f"base rate {y_te.mean():.3f}")
ax.scatter(m_te_05["recall"], m_te_05["precision"], color="grey", zorder=5, label="test @ 0.50")
ax.scatter(m_te_thr["recall"], m_te_thr["precision"], color="red", zorder=5, label=f"test @ {thr:.2f}")
ax.set(title="Precision–recall curve", xlabel="Recall", ylabel="Precision", ylim=(0, 1.02))
ax.legend(loc="upper right")

# Reliability (calibrated)
ax = axes[0, 2]
for label, yy, pp, ls in (("test", y_te, p_te, "-"), ("train OOF", y_tr, p_oof, "--")):
    frac_pos, mean_pred = calibration_curve(yy, pp, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, "o" + ls, label=label)
ax.plot([0, 1], [0, 1], ":", color="grey", label="perfect calibration")
ax.set(title=f"Reliability — test Brier {m_te_05['brier']:.3f}, ECE {m_te_05['ece']:.3f}",
       xlabel="Mean predicted probability", ylabel="Observed positive rate")
ax.legend(loc="upper left")

# Probability distribution by class
ax = axes[1, 0]
bins = np.linspace(0, 1, 41)
ax.hist(p_te[y_te == 0], bins=bins, alpha=0.6, density=True, label="y = 0")
ax.hist(p_te[y_te == 1], bins=bins, alpha=0.6, density=True, label="y = 1")
ax.axvline(0.5, color="grey", ls=":", label="0.50")
ax.axvline(thr, color="red", ls="--", label=f"CV threshold {thr:.2f}")

## Cell 14 — Tune and Evaluate the Stage 1 Threshold

Selects the F2-optimal decision threshold from the **training OOF probabilities**, then applies it to both OOF train and held-out test probabilities.

The cell:

* tunes the threshold using `y_train` and `proba_train_oof`;
* updates the Stage 1 threshold and predictions;
* compares metrics at 0.50 and the tuned threshold;
* plots training OOF F2 across the threshold sweep.

Test labels are used only for evaluation, not threshold selection.

In [ ]:
# ============================================================
# CELL 14: STAGE 1 DECISION THRESHOLD         
# ============================================================
# The in-stage F2 sweep chosen during fit (0.05–0.49 step 0.01, ties → lowest)
# — the same procedure GLASS uses (audit F7).
from blackbox_pipeline.models.mlp.evaluation import metrics_table
STAGE1_THRESHOLD = stage1.optimal_threshold
STAGE1_THRESHOLD_SOURCE = stage1.THRESHOLD_SOURCE

STAGE1_OUTPUT["threshold"] = STAGE1_THRESHOLD
STAGE1_OUTPUT["threshold_source"] = STAGE1_THRESHOLD_SOURCE
STAGE1_OUTPUT["pred_train_oof"] = (STAGE1_OUTPUT["proba_train_oof"] >= STAGE1_THRESHOLD).astype("int8")
STAGE1_OUTPUT["pred_test"] = (STAGE1_OUTPUT["proba_test"] >= STAGE1_THRESHOLD).astype("int8")

stage1_metrics = pd.concat([
    metrics_table(STAGE1["y_train"], STAGE1_OUTPUT["proba_train_oof"], "train OOF",
                  thresholds=(0.5, STAGE1_THRESHOLD), calibration_method=stage1.calibration_method),
    metrics_table(STAGE1["y_test"], STAGE1_OUTPUT["proba_test"], "test",
                  thresholds=(0.5, STAGE1_THRESHOLD), calibration_method=stage1.calibration_method),
])

print(f"threshold: {STAGE1_THRESHOLD:.3f}  [{STAGE1_THRESHOLD_SOURCE}]  (CV F2 {stage1.cv_f2:.4f})\n")
print(stage1_metrics.round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(stage1.threshold_sweep["threshold"], stage1.threshold_sweep["f_beta"],
        color="#185FA5", lw=2, label="train OOF F2")
ax.axvline(STAGE1_THRESHOLD, color="#3d3d3a", ls="--", lw=1.5, label=f"threshold {STAGE1_THRESHOLD:.2f}")
ax.axvline(0.5, color="#b4b2a9", ls=":", lw=1.5, label="0.5")
ax.set(title="Stage 1 MLP — F2 vs threshold (training OOF)", xlabel="Threshold", ylabel="F2")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

In [ ]:
# ============================================================
# CELL 14A: VALIDATION SETUP — STAGE 1 HAND-OFF OBJECT
# ============================================================
from pathlib import Path

from shared.pinned_artifacts import pinned_path
from shared.stage_io import StageOutput, split_fingerprint
from shared.validation import ValidationReport
from blackbox_pipeline.models.mlp import save_stage1_mlp

VALIDATION = ValidationReport()

# Current split identity
_TRAIN_IDX = GLOBAL_SPLIT["X_train"].index
_TEST_IDX = GLOBAL_SPLIT["X_test"].index
_Y_TR = GLOBAL_SPLIT["y_train"]
_Y_TE = GLOBAL_SPLIT["y_test"]

_FP = split_fingerprint(_TRAIN_IDX, _TEST_IDX, _Y_TR, _Y_TE)
SPLIT_FP = _FP

# Save black-box Stage 1 outputs
MLP_S1 = stage1.to_stage_output(
    STAGE1["X_test"], STAGE1["y_test"],
    threshold=STAGE1_THRESHOLD,
    threshold_source=STAGE1_THRESHOLD_SOURCE,
    metrics_test=stage1_metrics.loc[
        f"test @ {STAGE1_THRESHOLD:.2f}"
    ].to_dict(),
)

# The Stage 1 hand-off Stage 4 reads is the immutable MLP version, which holds the
# StageOutput (PR 39, execution contract §7.1). The fixed-name StageOutput file earlier
# runs wrote (models/mlp/stage1_blackbox_output.joblib, which the PR 36-38 arbiters record
# consuming) is no longer written.
MLP_ARTIFACT_PATH = save_stage1_mlp(
    stage1, "./models/mlp",
    stage_output=MLP_S1,
    test_metrics=MLP_S1.metrics_test,
)
MLP_S1_PATH = MLP_ARTIFACT_PATH

# The GLASS Stage 1 artifact this run pinned (PR 39, F9): the file GLASS Cell 8 wrote,
# by path and SHA-256 from $BANK_UPSTREAM_PINS, never the newest file in models/lr.
GLASS_S1_PATH = pinned_path("stage1.lr")
GLASS_S1 = StageOutput.load(GLASS_S1_PATH)
if GLASS_S1.split_fingerprint != SPLIT_FP:
    raise ValueError(f"{GLASS_S1_PATH} is from split {GLASS_S1.split_fingerprint}, "
                     f"not this notebook's {SPLIT_FP}")

STAGE1_OUTPUT["split_fingerprint"] = MLP_S1.split_fingerprint

assert MLP_S1.split_fingerprint == GLASS_S1.split_fingerprint == SPLIT_FP

print(f"split fingerprint : {_FP} (GLASS: {GLASS_S1.split_fingerprint})")
print(f"Stage 1 contract  : {MLP_S1}")
print(f"saved             : {MLP_S1_PATH}")
print(f"artifact          : {MLP_ARTIFACT_PATH}")
print(f"GLASS artifact    : {GLASS_S1_PATH}")

## Cell 15 — Stage 2 experiment constants

Defines the RF parameter source and tuning budget used by later cells.

* `"artifact"` — load saved RF parameters from `RF_ARTIFACT_DIRS`.
* `"tune"` — run `RF_TUNE_TRIALS` fresh trials.
* `"explicit"` — use `RF_PARAMS_EXPLICIT` directly.

`RF_PASS2_PARAMS = None` reuses Pass 1 hyperparameters for Pass 2. Separate Pass 2 tuning is valid only on its own OOF remainder, not against reported results.

In [ ]:
# ============================================================
# CELL 15: STAGE 2 EXPERIMENT CONSTANTS
# ============================================================

RF_PARAM_SOURCE = "artifact"          # "artifact" | "tune" | "explicit"

# Read only by the legacy Cell 19 (RUN_LEGACY_STAGE2): the folder GLASS Cell 16 writes
# (save_rf_artifact); the pre-reorganisation folders are no longer searched (PR 36 F23).
# The GLASS Router bundle is not searched for: Cell 18 loads it by pinned path (PR 39).
RF_ARTIFACT_DIRS = [BANK_ROOT / "models" / "rf"]

RF_TUNE_TRIALS = 100

RF_PARAMS_EXPLICIT = {
    "n_estimators": 500,
    "max_depth": None,
    "min_samples_leaf": 20,
    "max_features": "sqrt",
    "minority_weight": 1.0,
    "class_weight": "balanced",
}

RF_PASS2_PARAMS = None

# Shared fold plan: 5 stratified folds, same seed as the GLASS cross-fit and Stage 3.
N_STAGE_FOLDS = 5

assert RF_PARAM_SOURCE in ("artifact", "tune", "explicit"), RF_PARAM_SOURCE
print(f"RF_PARAM_SOURCE = {RF_PARAM_SOURCE!r}")

## Cell 16 — Import Stage 2 feature engineering

Imports `glass_router` RF feature engineering and fixes the shared 29-bin representation.

The assertion ensures Cell 4's Stage 1 `engineer_features` was not accidentally replaced by the RF version. If it fails, re-run Cell 4.

`RF_FEATURES_BINARY` must contain exactly 29 binary features, matching the representation used by both arms.

In [ ]:
# ============================================================
# CELL 16: IMPORT STAGE 2 FEATURE ENGINEERING 
# ============================================================
import sys

from shared.stage2 import feature_engineering as _glass_fe
from shared.stage2 import binning as _glass_binning

engineer_rf_features = _glass_fe.engineer_features
RF_SOURCE_FEATURES = list(_glass_fe.RF_SOURCE_FEATURES)
RF_FEATURES_BINARY = list(_glass_binning.RF_FEATURES_BINARY)

print(f"engineer_rf_features ← {_glass_fe.__file__}")
print(f"RF_FEATURES_BINARY   : {len(RF_FEATURES_BINARY)} bins")
assert len(RF_FEATURES_BINARY) == 29, "Expected the locked 29-bin feature list"

# Guard against the collision: Stage 1 must still be the LR engineer.
assert engineer_features is not engineer_rf_features, (
    "`engineer_features` (Stage 1 / LR) was overwritten by the RF version — "
    "re-run Cell 4."
)

## Cell 17 — Build Stage 2 binary features

Fits RF feature engineering on train, applies it to train and test, and stores the result in `GLASS_ROUTER_DATA`.

Assertions verify row alignment, exact feature names and order, binary values, and disjoint train/test indices.

Cell 18 repeats key split checks intentionally so failures can be localized to either feature engineering or routing.

In [ ]:
# ============================================================
# CELL 17: BUILD STAGE 2 BINARY FEATURES (GLASS_ROUTER_DATA)
# ============================================================
X_eng_train, X_eng_test = quiet(
    engineer_rf_features,
    GLOBAL_SPLIT["X_train"],
    GLOBAL_SPLIT["X_test"],
    GLOBAL_SPLIT["y_train"],
)

GLASS_ROUTER_DATA = {
    "X_eng_train": X_eng_train,
    "X_eng_test": X_eng_test,
    "y_eng_train": GLOBAL_SPLIT["y_train"],
    "y_eng_test": GLOBAL_SPLIT["y_test"],
}

assert X_eng_train.index.equals(GLOBAL_SPLIT["X_train"].index)
assert X_eng_test.index.equals(GLOBAL_SPLIT["X_test"].index)
assert list(X_eng_train.columns) == RF_FEATURES_BINARY
assert X_eng_train.isin([0, 1]).all().all() and X_eng_test.isin([0, 1]).all().all()
assert len(X_eng_train.index.intersection(X_eng_test.index)) == 0

print(f"GLASS_ROUTER_DATA: train {X_eng_train.shape} | test {X_eng_test.shape}")
print(f"train base rate: {GLASS_ROUTER_DATA['y_eng_train'].mean():.4f}")

## Cell 18 — Import the router and verify the shared split

Imports the Stage 2 router and computes `SPLIT_FP`, a fingerprint of the train/test row indices.

The fingerprint is stored with fitted routers and checked on reload, preventing evaluation against a different split that may contain training rows.

`unpack_global_split` also confirms the engineered frames preserve the original row identities and order.

In [ ]:
# ============================================================
# CELL 18: STAGE 2 INPUTS: shared split, shared fold plan, the pinned Stage 2 routers
# (replaces former Cells 18 + 18B)
# ============================================================
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from blackbox_pipeline.models.rf_router import (
    OperatingConstraints,
    RFRouter,
    RFRouterConfig,
    SingleRFRouter,
    crossfit_rf_router,
    load_rf_router,
    save_rf_router,
    split_fingerprint,
    unpack_global_split,
)
from blackbox_pipeline.models.rf_router.evaluation.router_metrics import (
    compare_routers,
    evaluate_router,
)
from glass_pipeline.glass_router.pipeline import fold_fingerprint, make_folds
from shared.notebook_switches import register_switch
from shared.pinned_artifacts import load_pinned_stage2
from shared.stage4 import build_stage4_inputs, router_stream

# PR 39 (execution contract §5.3): Stage 2 is not fitted in this notebook. Both routers
# are the accepted PR 38 run's, by pinned path and SHA-256. Cells 19-28B (the legacy
# notebook RF Router and the PR 35 experiment) run only when RUN_LEGACY_STAGE2 is True,
# with LEGACY_GLASS_BUNDLE_PATH naming the bundle GLASS Cell 16 saved.
RUN_LEGACY_STAGE2 = False
LEGACY_GLASS_BUNDLE_PATH = None
register_switch("legacy_stage2", "RUN_LEGACY_STAGE2")

STAGE2 = unpack_global_split(GLOBAL_SPLIT, engineered=GLASS_ROUTER_DATA)
SPLIT_FP = STAGE2["split_fingerprint"]

# One fold plan for everything that produces train-side OOF outputs. Same
# StratifiedKFold(shuffle=True, random_state) as crossfit_stage2 and
# rf_router.oof.make_folds; Stage 2 · E asserts the routers actually used it.
STAGE_FOLDS = make_folds(STAGE2["y_train"], N_STAGE_FOLDS, RANDOM_STATE)
STAGE_FOLD_ID = np.full(len(STAGE2["y_train"]), -1)
for _k, (_tr, _va) in enumerate(STAGE_FOLDS):
    STAGE_FOLD_ID[_va] = _k
STAGE_FOLD_FP = fold_fingerprint(STAGE_FOLD_ID)

if RUN_LEGACY_STAGE2:
    if not LEGACY_GLASS_BUNDLE_PATH:
        raise ValueError("RUN_LEGACY_STAGE2 needs LEGACY_GLASS_BUNDLE_PATH: the "
                         "models/glass_router/glass_router_*.joblib GLASS Cell 16 saved")
    GLASS_BUNDLE_PATH = Path(LEGACY_GLASS_BUNDLE_PATH)
    GLASS_BUNDLE = joblib.load(GLASS_BUNDLE_PATH)
else:
    STAGE2_PINNED = load_pinned_stage2(GLOBAL_SPLIT)  # acceptance and Stage 4 contract checked
    GLASS_BUNDLE, GLASS_BUNDLE_PATH = STAGE2_PINNED.glass_bundle, STAGE2_PINNED.glass_path
    RF_ROUTER_PATH = str(STAGE2_PINNED.rf_path)      # the RF Router Cells 35-37 read
    print(STAGE2_PINNED.describe())
if not pd.Index(GLASS_BUNDLE["index_test"]).equals(STAGE2["X_test"].index):
    raise AssertionError("GLASS test outputs are not in this notebook's X_test order")
if GLASS_BUNDLE["train_outputs_oof"] is not True:
    raise AssertionError("GLASS train-side outputs are not OOF")
_glass_fold_fp = (GLASS_BUNDLE.get("oof_protocol") or {}).get("fold_fingerprint")

# The GLASS bundle sets the RF Router's operating constraints (Cell 19), so it must pass
# the Stage 4 router contract now, not at the crossover in Cell 37: label fingerprints
# for both sides (F12), declared Pass 2 populations (F17), split / fold provenance.
_glass_stream, _glass_checks = router_stream(
    GLASS_BUNDLE, "glass", "GLASS Router", STAGE2["y_train"].to_numpy())
build_stage4_inputs(GLOBAL_SPLIT, [_glass_stream], _glass_checks, verbose=False)

print(f"split fingerprint : {SPLIT_FP}")
print(f"fold fingerprint  : {STAGE_FOLD_FP}  ({N_STAGE_FOLDS} folds)")
print(f"train / test      : {STAGE2['n_train']:,} / {STAGE2['n_test']:,}")
print(f"rf_router         ← {sys.modules[RFRouter.__module__].__file__}")
print(f"GLASS artifact    ← {GLASS_BUNDLE_PATH}")
print(f"GLASS evidence    : labels — {_glass_stream.notes['label_evidence']}; "
      f"Pass 2 — {_glass_stream.notes['pass2_population_evidence']}")
print(f"GLASS fold fp     : {_glass_fold_fp} "
      f"({'MATCH' if _glass_fold_fp == STAGE_FOLD_FP else 'MISMATCH — folds differ'})")


In [ ]:
%%legacy_stage2
# ============================================================
# CELL 19: STAGE 2 CONFIGURATION: set-level operating constraints + baseline RF params
# (replaces former Cells 19, 20, 21, 22, 23; tuning/loading logic now lives in
#  blackbox_pipeline.models.rf_router.tuning.legacy, behaviour unchanged)
# ============================================================
from blackbox_pipeline.models.rf_router.tuning import resolve_rf_params

CONSTRAINT_SOURCE = "match_glass_oof"

GLASS_OOF_METRICS = evaluate_router(
    GLASS_BUNDLE["train_pred"],
    GLASS_BUNDLE["train_confidence"],
    GLASS_BUNDLE["train_decisions"],
    STAGE2["y_train"],
    label="GLASS Router (train OOF)",
)

if CONSTRAINT_SOURCE == "match_glass_oof":
    _m = GLASS_OOF_METRICS
    CONSTRAINTS = OperatingConstraints(
        min_precision_not_subscribe=float(_m["pass1_npv"]),
        max_subscriber_leakage_rate=float(_m["pass1_leakage_rate"]),
        max_subscriber_leakage_absolute=int(_m["pass1_subscribers_lost"]),
        min_pass1_coverage=None,
        min_precision_subscribe=float(_m["pass2_precision"]),
        min_recall_subscribe=None,
        recall_denominator="global",
        min_band_support_pass1=125,
        min_band_support_pass2=75,
    )
else:
    CONSTRAINTS = OperatingConstraints(
        min_precision_not_subscribe=0.95,
        max_subscriber_leakage_rate=0.25,
        max_subscriber_leakage_absolute=950,
        min_pass1_coverage=None,
        min_precision_subscribe=0.42,
        min_recall_subscribe=None,
        recall_denominator="global",
        min_band_support_pass1=125,
        min_band_support_pass2=75,
    )
print(f"CONSTRAINT_SOURCE = {CONSTRAINT_SOURCE!r}")
print(CONSTRAINTS)

# Baseline hyperparameters. Only the branch RF_PARAM_SOURCE selects reads its
# inputs, so undefined names for the other branches are fine.
RF_PARAMS, RF_PARAMS_ORIGIN = resolve_rf_params(
    RF_PARAM_SOURCE,
    split_fp=SPLIT_FP,
    artifact_dirs=globals().get("RF_ARTIFACT_DIRS"),
    X_train=STAGE2["X_train"],
    y_train=STAGE2["y_train"],
    n_trials=globals().get("RF_TUNE_TRIALS"),
    explicit=globals().get("RF_PARAMS_EXPLICIT"),
    random_state=RANDOM_STATE,
)
print("\nPass 2 forest:",
      "inherits Pass 1 params" if RF_PASS2_PARAMS is None else "separate params")


In [ ]:
%%legacy_stage2
# ============================================================
# CELL 20: STAGE 2 EXPERIMENTS 1 & 2: single-RF ablation + two-pass RF router
# (replaces former Cells 24 + 25) — training split only
# ============================================================
from dataclasses import replace

# Experiment 1 — single-RF two-threshold ablation.
# At the legacy GLASS operating point (≈54% leakage, ≈27% Pass 2 precision) the
# two bands on one RF score overlap (t1 > t2): there is no abstain region, the
# same as GLASS legacy (≈0.1% abstain). Recorded as a finding; ordering check
# relaxed for the ablation only. Pass 1 still claims first.
ABLATION_CONSTRAINTS = replace(CONSTRAINTS, require_band_ordering=False)
ablation = SingleRFRouter(RFRouterConfig(
    mode="single_rf",
    constraints=ABLATION_CONSTRAINTS,
    rf_params=RF_PARAMS,
    n_oof_folds=N_STAGE_FOLDS,
    random_state=RANDOM_STATE,
    verbose=True,
)).fit(STAGE2["X_train"], STAGE2["y_train"], SPLIT_FP)
if ablation.thresholds_.t1 > ablation.thresholds_.t2:
    print(f"\n⚠️  FINDING: bands overlap (t1={ablation.thresholds_.t1:.4f} > "
          f"t2={ablation.thresholds_.t2:.4f}) — no abstain region at this operating point.")

# Experiment 2 — two-pass RF router (the GLASS counterpart).
VALIDATE_NESTED = True


def make_router_config(validate_nested=False, verbose=True):
    return RFRouterConfig(
        mode="two_pass",
        constraints=CONSTRAINTS,
        rf_params=RF_PARAMS,
        pass2_rf_params=RF_PASS2_PARAMS,
        n_oof_folds=N_STAGE_FOLDS,
        n_inner_folds=5,
        random_state=RANDOM_STATE,
        remainder_min_size=500,
        validate_nested=validate_nested,
        verbose=verbose,
    )


router = RFRouter(make_router_config(VALIDATE_NESTED)).fit(
    STAGE2["X_train"], STAGE2["y_train"], SPLIT_FP,
)
if router.fit_report_.nested_check:
    print("\nNested cascade check:")
    print(pd.Series(router.fit_report_.nested_check).to_string())

In [ ]:
%%legacy_stage2
# ============================================================
# CELL 21: Stage 2 OUTER CROSS-FIT (Stage 4 train-side inputs)
# (replaces former Cell 25B; loop now in rf_router.crossfit, behaviour unchanged)
#
# Per shared fold: engineer_features fit on the other folds -> bins -> full
# RFRouter fit (own inner OOF + thresholds) -> route ONLY the held-out fold.
# ============================================================
CROSSFIT = crossfit_rf_router(
    GLOBAL_SPLIT["X_train"],
    STAGE2["y_train"],
    STAGE_FOLDS,
    engineer=lambda X_tr, X_va, y_tr: quiet(engineer_rf_features, X_tr, X_va, y_tr),
    make_config=lambda: make_router_config(validate_nested=False, verbose=False),
    split_fp=SPLIT_FP,
    fold_fp=STAGE_FOLD_FP,
    random_state=RANDOM_STATE,
)
RF_OOF, RF_OOF_FOLDS, RF_OOF_PROTOCOL = CROSSFIT.oof, CROSSFIT.folds, CROSSFIT.protocol

# Diagnostic: how different is the cheaper predict_train_oof path?
_p_cheap, _, _d_cheap = router.predict_train_oof()
print("decision agreement, outer cross-fit vs predict_train_oof: "
      f"{(_d_cheap == RF_OOF['decisions']).mean():.4f}")


In [ ]:
%%legacy_stage2
# ============================================================
# CELL 22: STAGE 2 TRAINING-OOF DIAGNOSTICS (still no test data)
# (replaces former Cells 26 + 27 + 28)
# ============================================================
import matplotlib.pyplot as plt

from blackbox_pipeline.models.rf_router.reporting import (
    feasible_tail,
    pass_agreement,
    plot_threshold_sweeps,
    train_oof_summary,
)

# Provenance: both routers used the shared split AND the shared fold plan.
assert ablation.is_fitted and router.is_fitted
assert ablation.split_fingerprint_ == router.split_fingerprint_ == SPLIT_FP
assert np.array_equal(router.oof_pass1_fold_id_, STAGE_FOLD_ID), "router OOF folds != STAGE_FOLDS"
assert np.array_equal(ablation.oof_fold_id_, STAGE_FOLD_ID), "ablation OOF folds != STAGE_FOLDS"

plot_threshold_sweeps(router, CONSTRAINTS)
plt.show()

print("Which constraint binds at the chosen band (largest feasible bands):")
print(feasible_tail(router.thresholds_.pass1.sweep).round(4).to_string(index=False))
print()
print(feasible_tail(router.thresholds_.pass2.sweep).round(4).to_string(index=False))

print("\nOperating points on training OOF:")
print(train_oof_summary({"single_rf": ablation, "two_pass": router}).round(4).to_string())

print("\nPass 1 vs Pass 2 agreement on the remainder:")
print(pd.Series(pass_agreement(router), dtype=object).to_string())
print("\n✅ both operating points frozen; split and fold fingerprints match")


In [ ]:
%%legacy_stage2
# ============================================================
# CELL 23: STAGE 2 COMPARISON: ALL ARMS, SAME METRIC CODE
# (replaces former Cell 29) — each router scores X_test exactly once here
# ============================================================
from blackbox_pipeline.models.rf_router.reporting import HEADLINE_METRICS as HEADLINE

assert split_fingerprint(STAGE2["X_train"], STAGE2["X_test"]) == SPLIT_FP
assert len(STAGE2["X_train"].index.intersection(STAGE2["X_test"].index)) == 0

# Each router scores X_test ONCE; Stage 2 · G reuses these outputs.
TEST_OUT = {
    "RF single (ablation)": ablation.predict_with_proba(STAGE2["X_test"]),
    "RF two-pass": router.predict_with_proba(STAGE2["X_test"]),
}

stage2_results = compare_routers(
    evaluate_router(
        GLASS_BUNDLE["test_pred"],
        GLASS_BUNDLE["test_confidence"],
        GLASS_BUNDLE["test_decisions"],
        STAGE2["y_test"],
        label="GLASS Router",
        base_rate=GLASS_BUNDLE["training_base_rate"],
    ),
    evaluate_router(*TEST_OUT["RF single (ablation)"][:3], STAGE2["y_test"],
                    label="RF single (ablation)", base_rate=ablation.training_base_rate),
    evaluate_router(*TEST_OUT["RF two-pass"][:3], STAGE2["y_test"],
                    label="RF two-pass", base_rate=router.training_base_rate),
)
print("Stage 2 — TEST split, set-level metrics")
print(stage2_results.loc[HEADLINE].round(4).to_string())

# Train OOF — what Stage 4 will actually consume.
stage2_oof_results = compare_routers(
    GLASS_OOF_METRICS | {"label": "GLASS Router"},
    evaluate_router(RF_OOF["pred"], RF_OOF["confidence"], RF_OOF["decisions"],
                    STAGE2["y_train"], label="RF two-pass"),
)
print(f"\nStage 2 — TRAIN OOF (outer cross-fit), set-level metrics | "
      f"constraints: {CONSTRAINT_SOURCE}")
print(stage2_oof_results.loc[HEADLINE].round(4).to_string())
print("\nRF two-pass outer cross-fit, per fold:")
print(RF_OOF_FOLDS.round(4).to_string(index=False))


In [ ]:
%%legacy_stage2
# ============================================================
# CELL 24: SAVE THE ROUTERS + PACKAGE STAGE 2 OUTPUTS FOR STAGE 3
# (replaces former Cells 30 + 31) — reuses TEST_OUT, no re-scoring
# ============================================================
STAGE2_OUT = "./models/rf_router"
_idx_tr, _idx_te = STAGE2["X_train"].index, STAGE2["X_test"].index
_p_te, _c_te, _d_te, _proba_te = TEST_OUT["RF two-pass"]
_common_meta = {
    "index_train": _idx_tr,
    "index_test": _idx_te,
    "constraint_source": CONSTRAINT_SOURCE,
    "rf_params_origin": RF_PARAMS_ORIGIN,
}

save_rf_router(
    ablation, STAGE2_OUT, name="rf_router_single",
    test_metrics=stage2_results["RF single (ablation)"].to_dict(),
    extra_metadata=_common_meta,
)
RF_ROUTER_PATH = save_rf_router(           # the immutable version (Cells 35-37 read it)
    router, STAGE2_OUT, name="rf_router_two_pass",
    test_metrics=stage2_results["RF two-pass"].to_dict(),
    extra_metadata={
        **_common_meta,
        "glass_artifact": str(GLASS_BUNDLE_PATH),
        # Stage 4 inputs — train side (outer cross-fit)
        "train_outputs_oof": True,
        "train_pred": RF_OOF["pred"],
        "train_confidence": RF_OOF["confidence"],
        "train_decisions": RF_OOF["decisions"],
        "train_proba": RF_OOF["proba"],
        "train_fold_id": STAGE_FOLD_ID,
        "oof_protocol": RF_OOF_PROTOCOL,
        "oof_fold_summary": RF_OOF_FOLDS.to_dict("records"),
        # Test side — full-train router
        "test_pred": _p_te,
        "test_confidence": _c_te,
        "test_decisions": _d_te,
        "test_proba": _proba_te,
    },
)
_reloaded = load_rf_router(RF_ROUTER_PATH, expected_split_fingerprint=SPLIT_FP)
assert _reloaded["extra"]["oof_protocol"]["fold_fingerprint"] == STAGE_FOLD_FP
print(f"\nreload OK — t1={_reloaded['thresholds']['t1']:.6f} "
      f"t2={_reloaded['thresholds']['t2']:.6f} | fold fp {STAGE_FOLD_FP}")


In [ ]:
%%legacy_stage2
# ============================================================
# CELL 25: STAGE 2 OUTPUT PACKAGING AND STAGE 3 HANDOFF
# ============================================================
STAGE2_OUTPUT = {
    "router": router,
    "ablation": ablation,
    "t1": router.thresholds_.t1,
    "t2": router.thresholds_.t2,
    # Train side: outer cross-fit — features, forests and thresholds all OOF.
    "preds_train_oof": pd.Series(RF_OOF["pred"], index=_idx_tr),
    "conf_train_oof": pd.Series(RF_OOF["confidence"], index=_idx_tr),
    "decisions_train_oof": pd.Series(RF_OOF["decisions"], index=_idx_tr),
    "proba_train_oof": RF_OOF["proba"],
    "fold_id_train": pd.Series(STAGE_FOLD_ID, index=_idx_tr),
    # Test side: full-train router
    "preds_test": pd.Series(_p_te, index=_idx_te),
    "conf_test": pd.Series(_c_te, index=_idx_te),
    "decisions_test": pd.Series(_d_te, index=_idx_te),
    "proba_test": _proba_te,
    "metrics_test": stage2_results,
    "metrics_train_oof": stage2_oof_results,
    "split_fingerprint": SPLIT_FP,
    "fold_fingerprint": STAGE_FOLD_FP,
}
STAGE2_OUTPUT["abstain_train"] = STAGE2_OUTPUT["decisions_train_oof"] == "uncertain"
STAGE2_OUTPUT["abstain_test"] = STAGE2_OUTPUT["decisions_test"] == "uncertain"

# OOF vs in-sample: compare full decisions, not only the abstain mask — at a
# near-zero-abstain operating point both masks are all False and match trivially.
_in_sample_d = router.predict(STAGE2["X_train"])[2]
_oof_d = STAGE2_OUTPUT["decisions_train_oof"].to_numpy()
_oof_ab = STAGE2_OUTPUT["abstain_train"].to_numpy()
_te_ab = STAGE2_OUTPUT["abstain_test"].to_numpy()
print(f"decision agreement, OOF vs in-sample: {(_oof_d == _in_sample_d).mean():.4f} "
      f"| abstains OOF {_oof_ab.sum():,} vs in-sample {(_in_sample_d == 'uncertain').sum():,}")
assert not np.array_equal(_oof_d, _in_sample_d), (
    "OOF and in-sample decisions identical — OOF path not being used"
)

print("\nStage 2 → Stage 3 handoff")
print(f"  train abstains (OOF): {_oof_ab.sum():,} ({_oof_ab.mean():.1%})")
print(f"  test  abstains      : {_te_ab.sum():,} ({_te_ab.mean():.1%})")
for _name, _mask, _y in (("train", _oof_ab, STAGE2["y_train"]), ("test ", _te_ab, STAGE2["y_test"])):
    print(f"  {_name} abstain base rate: "
          + (f"{_y[_mask].mean():.4f}" if _mask.any() else "n/a (no abstains)"))
if not _oof_ab.any() or not _te_ab.any():
    print("\n⚠️  No (or almost no) abstains at this operating point (legacy-matched) — "
          "Stage 3 would receive an empty population; see PR 33.")

In [ ]:
%%legacy_stage2
# ============================================================
# CELL 26: OBJECTIVE SPECIALIZATION — Gate 0 + Gate 1 screen (training only)
#
# Question: does each pass do better tuned for its own decision than for the
# shared ROC-AUC objective the baseline used?
#   Pass 1 objective: coverage at the shared constraints (NPV floor + leakage caps)
#   Pass 2 objective: recall at the precision floor, on the Pass 1 remainder
#
# Gate 0 (free): if the remainder's base rate already clears the precision floor,
#   Pass 2 flags the whole remainder whatever its hyperparameters — Pass 2
#   tuning cannot change routing, and Cell 27 skips those runs.
# Gate 1 (SPEC_RUN_PROBE): a SCREEN — random configs scored on both passes plus
#   a noise floor. "LIMITED OBSERVED HEADROOM" on every question means this
#   probe found nothing distinguishable from noise; stopping there is a
#   reasonable time-boxed call, but report it as a screening result, not as
#   evidence that specialization cannot help. "INSUFFICIENT EVIDENCE" means the
#   probe or the noise floor was too small to say anything.
# ============================================================
from blackbox_pipeline.models.rf_router.tuning import (
    Pass1Stage,
    RouterFolds,
    TuningBudget,
    cache_key,
    cached,
    data_fingerprint,
    diagnose_router,
    frame_validator,
    headroom_probe,
    noise_floor,
    summarize_headroom,
)

SPEC_DIR = Path(STAGE2_OUT) / "specialization"
# PR 35 is concluded (no effect detected; A0 kept — research_logs/
# pr35_rf_router_objective_specialization.md). The cells stay for reproducibility but are
# off by default, so a clean-kernel run neither re-tunes nor evaluates the test split again
# (each evaluation is logged by register_test_evaluation). Set True only to reproduce PR 35.
SPEC_RUN_PROBE = False      # Gate 1: ~SPEC_PROBE_CONFIGS × 2 × N_STAGE_FOLDS forest fits
SPEC_RUN_TUNING = False     # Cell 27: 5 × n_trials × N_STAGE_FOLDS forest fits + arm fits
SPEC_EVALUATE_TEST = False  # Cell 28: evaluates the test split — set after Cell 27 is frozen
SPEC_PROBE_CONFIGS = 40
SPEC_NOISE_SEEDS = (0, 1, 2, 3, 4)

# Facts about X that its content cannot reveal. The cache key already hashes the
# ordered schema, dtypes, row index, every feature value and every label; add
# anything else that identifies the feature block (e.g. a binning version).
SPEC_DATA_PROVENANCE = {"feature_source": "GLASS_ROUTER_DATA['X_eng_train']"}
SPEC_DATA_FP = data_fingerprint(STAGE2["X_train"], STAGE2["y_train"],
                                provenance=SPEC_DATA_PROVENANCE)

# ONE budget for every arm. fold_seed differs from the router's RANDOM_STATE so
# the router re-scores each selected config on folds it was not selected on.
# That reduces fold-assignment optimism; it does NOT remove selection bias,
# because the same training labels are reused — training-OOF arm comparisons
# below are diagnostic only.
SPEC_BUDGET = TuningBudget(
    n_trials=40,
    sampler_seed=RANDOM_STATE,
    n_folds=N_STAGE_FOLDS,
    fold_seed=RANDOM_STATE + 1,
    forest_seed=RANDOM_STATE,
    fixed_params={"n_estimators": 500},
)
SPEC_ROUTER_FOLDS = RouterFolds.from_config(make_router_config(verbose=False))

# ---- Gate 0 — free read-out of the fitted baseline router -------------------
SPEC_GATE0 = diagnose_router(router)
print("Gate 0 — baseline operating point (training OOF)")
print(pd.Series({k: v for k, v in SPEC_GATE0.items() if k != "verdict"},
                dtype=object).to_string())
print(f"\n{SPEC_GATE0['verdict']}")

SPEC_BASE_STAGE = Pass1Stage.from_router(router)  # A0's remainder, no refit
SPEC_NOISE = SPEC_PROBE = SPEC_HEADROOM = None

# ---- Gate 1 — headroom screen + noise floor ------------------------------------
if SPEC_RUN_PROBE:
    _parts = dict(budget=SPEC_BUDGET.signature(), constraints=CONSTRAINTS.to_dict(),
                  params=RF_PARAMS, remainder=SPEC_BASE_STAGE.remainder_fingerprint)
    _noise_params = {**RF_PARAMS, **SPEC_BUDGET.fixed}  # same n_estimators as the arms
    SPEC_NOISE = cached(
        SPEC_DIR, "noise_floor",
        cache_key("noise_floor", data_fp=SPEC_DATA_FP, split_fp=SPLIT_FP,
                  seeds=list(SPEC_NOISE_SEEDS), **_parts),
        lambda: noise_floor(_noise_params, STAGE2["X_train"], STAGE2["y_train"],
                            constraints=CONSTRAINTS, budget=SPEC_BUDGET,
                            stage=SPEC_BASE_STAGE, seeds=SPEC_NOISE_SEEDS),
        validate=frame_validator(["seed", "p1_coverage", "p2_recall"],
                                 n_rows=len(SPEC_NOISE_SEEDS)),
    )
    SPEC_PROBE = cached(
        SPEC_DIR, "headroom_probe",
        cache_key("headroom_probe", data_fp=SPEC_DATA_FP, split_fp=SPLIT_FP,
                  n_configs=SPEC_PROBE_CONFIGS, **_parts),
        lambda: headroom_probe(STAGE2["X_train"], STAGE2["y_train"],
                               constraints=CONSTRAINTS, budget=SPEC_BUDGET,
                               stage=SPEC_BASE_STAGE, n_configs=SPEC_PROBE_CONFIGS),
        validate=frame_validator(["p1_auc", "p1_coverage", "rem_auc", "p2_recall"],
                                 n_rows=SPEC_PROBE_CONFIGS),
    )
    SPEC_HEADROOM = summarize_headroom(SPEC_PROBE, SPEC_NOISE)
    print("\nTraining-side noise floor (fresh fold + forest seeds):")
    print(SPEC_NOISE.drop(columns="seed").agg(["mean", "std"]).round(4).to_string())
    print("\nGate 1 — headroom SCREEN (what this probe observed; not a test):")
    print(SPEC_HEADROOM.drop(columns="evidence").round(4).to_string())
    for _q, _row in SPEC_HEADROOM.iterrows():
        print(f"  • {_q}: {_row['screening']} — {_row['evidence']}")
else:
    print("\nGate 1 skipped (SPEC_RUN_PROBE = False).")

In [ ]:
%%legacy_stage2
# ============================================================
# CELL 27: OBJECTIVE SPECIALIZATION — tune the arms (one budget) + fit arm routers (training only)
#
#   A0 current baseline         RF_PARAMS both passes (the router from Cell 20)
#   B  auc | inherit            AUC re-tuned in this harness, both passes
#   C  auc | auc@remainder      B's Pass 1; Pass 2 AUC on B's remainder (control)
#   D  auc | recall@remainder   B's Pass 1; Pass 2 recall@precision on B's remainder
#   E  coverage | recall@rem.   Pass 1 coverage@constraints; Pass 2 recall on E's remainder
#
# Pass 1 is tuned first; each remainder is rebuilt with the ROUTER'S folds and
# frozen; Pass 2 is tuned on it and stamped with its fingerprint; fit_arm()
# re-checks the stamp against the fitted router. C and D are verified to share
# the frozen remainder, budget, trial count and data (matched control).
# Degenerate remainders skip their Pass 2 runs. Results are cached under
# SPEC_DIR keyed on the data fingerprint and tuning contract version; a cached
# file that does not match the current inputs raises instead of being reused.
# ============================================================
from blackbox_pipeline.models.rf_router.reporting import pass_agreement, train_oof_summary
from blackbox_pipeline.models.rf_router.tuning import ARM_A0, fit_arm, run_specialization_tuning

SPEC_PLAN, SPEC_ROUTERS = None, {}
if SPEC_RUN_TUNING:
    SPEC_PLAN = run_specialization_tuning(
        STAGE2["X_train"],
        STAGE2["y_train"],
        constraints=CONSTRAINTS,
        budget=SPEC_BUDGET,
        router_folds=SPEC_ROUTER_FOLDS,
        split_fp=SPLIT_FP,
        baseline_params=RF_PARAMS,
        baseline_pass2_params=RF_PASS2_PARAMS,
        data_provenance=SPEC_DATA_PROVENANCE,
        cache_dir=SPEC_DIR,
        reevaluate_seeds=(0, 1, 2),
    )
    print("\nProvenance:", SPEC_PLAN.provenance)
    print("\nArms:")
    print(SPEC_PLAN.arm_table().to_string())
    for _check, _status in SPEC_PLAN.controls.items():
        print(f"  control check — {_check}: {_status}")
    for _arm, _why in SPEC_PLAN.skipped.items():
        print(f"  skipped {_arm}: {_why}")

    _tt = SPEC_PLAN.tuning_table()
    _tt.to_csv(SPEC_DIR / "tuning_runs.csv")
    print("\nTuning runs (training labels — diagnostic). best_value is selection-")
    print("optimistic; fold_optimism is only its fold-assignment part (a lower bound):")
    print(_tt[["objective", "pass", "best_value", "fresh_mean", "fresh_sd",
               "fold_optimism", "n_infeasible"]].round(4).to_string())
    print("\nChosen hyperparameters per tuning run (mechanism):")
    print(_tt.filter(like="param:").to_string())

    _base_cfg = make_router_config(validate_nested=False, verbose=False)
    SPEC_ROUTERS = {ARM_A0: router}  # baseline router from Cell 20 — not refitted
    for _name, _arm in SPEC_PLAN.arms.items():
        if _name != ARM_A0:
            SPEC_ROUTERS[_name] = fit_arm(_arm, STAGE2["X_train"], STAGE2["y_train"],
                                          SPLIT_FP, _base_cfg)

    # Tuned arms were SELECTED on these labels, so training OOF flatters them.
    print("\nArms on training OOF — DIAGNOSTIC ONLY (selection-biased; not evidence):")
    print(train_oof_summary(SPEC_ROUTERS).round(4).to_string())
    print("\nPass 1 vs Pass 2 agreement per arm (router cut semantics, exact top-k):")
    print(pd.DataFrame({n: pass_agreement(r) for n, r in SPEC_ROUTERS.items()}).to_string())
else:
    print("Tuning skipped (SPEC_RUN_TUNING = False).")

In [ ]:
%%legacy_stage2
# ============================================================
# CELL 28: OBJECTIVE SPECIALIZATION — frozen arms on the test split + pre-registered verdict
#
# What this evidence is: the test split was examined in earlier Stage 2
# research, so this is a held-out comparison under criteria frozen BEFORE it
# runs — not untouched, confirmatory evidence. Do not revise arms, budgets or
# the rule because of what prints here; register_test_evaluation() refuses a
# changed rule and logs every evaluation of this split.
#
# The rule has two parts, both required to recommend an arm:
#   statistical — the paired-bootstrap 95% CI excludes 0, the effect clears
#     max(min_effect, 2 × training noise SD), and both arms' evaluations are
#     valid (no undefined constraint);
#   eligibility — the new arm satisfies EVERY applicable hard constraint on the
#     test split (NPV, leakage rate and absolute cap, coverage floor if set, band
#     support, precision, recall floor if set), regardless of how the control
#     does. Count constraints are held to the same proportion of the training
#     split they were solved on.
# "No effect detected" means nothing larger than the CI half-width could
# resolve — not that none exists.
#
# Stage 4 dependency (not changed here): Cell 21 cross-fits the router with
# hyperparameters selected once on all of X_train, so each held-out fold's labels
# took part in choosing them. That dependence is weak under ROC-AUC tuning and
# stronger when the tuning objective is the Stage 2 operating quantity itself.
# If a specialized design is adopted, the tuning must be repeated inside each
# outer fold before the cascade is regenerated.
# ============================================================
from blackbox_pipeline.models.rf_router.evaluation import paired_bootstrap, slippage_table
from blackbox_pipeline.models.rf_router.tuning import (
    DecisionRule,
    decomposition_pairs,
    register_test_evaluation,
    specialization_verdict,
)

SPEC_RULE = DecisionRule(min_effect=0.01, noise_multiplier=2.0, min_finite_boot=0.95)

if SPEC_EVALUATE_TEST and SPEC_ROUTERS:
    assert split_fingerprint(STAGE2["X_train"], STAGE2["X_test"]) == SPLIT_FP
    assert all(r.split_fingerprint_ == SPLIT_FP for r in SPEC_ROUTERS.values())

    _prior = register_test_evaluation(SPEC_DIR, SPEC_RULE, arms=list(SPEC_ROUTERS),
                                      split_fp=SPLIT_FP)
    print(f"Decision rule frozen in {SPEC_DIR / 'decision_rule.json'}; "
          f"{_prior} earlier evaluation(s) of this test split logged by this experiment "
          "(plus the earlier Stage 2 research, which is not logged here).")

    # A0 reuses Cell 23's test outputs; every other arm scores X_test once.
    SPEC_TEST_OUT = {
        n: TEST_OUT["RF two-pass"][:3] if n == ARM_A0 else r.predict(STAGE2["X_test"])
        for n, r in SPEC_ROUTERS.items()
    }
    SPEC_TEST = compare_routers(
        evaluate_router(GLASS_BUNDLE["test_pred"], GLASS_BUNDLE["test_confidence"],
                        GLASS_BUNDLE["test_decisions"], STAGE2["y_test"],
                        label="GLASS Router"),
        *(evaluate_router(*out, STAGE2["y_test"], label=n) for n, out in SPEC_TEST_OUT.items()),
    )
    print("\nObjective specialization — TEST split, set-level metrics")
    print(SPEC_TEST.loc[HEADLINE].round(4).to_string())

    # Count constraints are translated from the population they were solved on.
    SPEC_SLIPPAGE = slippage_table(SPEC_TEST, CONSTRAINTS,
                                   reference_rows=int(STAGE2["n_train"]),
                                   reference_positives=int(STAGE2["y_train"].sum()))
    print("\nHard constraints on test (margin > 0 = violation; statuses: ok / violated /")
    print("disabled = not configured / empty_band = nothing routed / undefined = invalid):")
    print(SPEC_SLIPPAGE.to_string())

    SPEC_BOOT = paired_bootstrap(
        {n: out[2] for n, out in SPEC_TEST_OUT.items()},
        STAGE2["y_test"],
        pairs=decomposition_pairs(list(SPEC_TEST_OUT)),
        n_boot=2000,
        seed=RANDOM_STATE,
    )
    SPEC_VERDICT = specialization_verdict(SPEC_BOOT, SPEC_SLIPPAGE, noise=SPEC_NOISE,
                                          rule=SPEC_RULE)
    print("\nEffect decomposition (paired bootstrap on test, 95% CI) under the frozen rule:")
    print(SPEC_VERDICT[["comparison", "metric", "delta", "ci_low", "ci_high", "ci_halfwidth",
                        "threshold", "statistical", "eligibility", "verdict"]]
          .round(4).to_string(index=False))
    if SPEC_NOISE is None:
        print("\n⚠️  No noise floor (Gate 1 not run): thresholds use min_effect only.")
    for _arm, _why in SPEC_PLAN.skipped.items():
        print(f"\nFinding — {_arm} not run: {_why}")

    for _name, _frame in (("test_metrics", SPEC_TEST), ("slippage", SPEC_SLIPPAGE),
                          ("bootstrap", SPEC_BOOT), ("verdict", SPEC_VERDICT)):
        _frame.to_csv(SPEC_DIR / f"{_name}.csv")
    print(f"\ntables written to {SPEC_DIR}")
else:
    print("Test evaluation skipped (needs SPEC_RUN_TUNING and SPEC_EVALUATE_TEST).")

In [ ]:
%%legacy_stage2
# ============================================================
# CELL 28B: OBJECTIVE SPECIALIZATION — arms C vs D on the test split (diagnostic)
# Reads Cell 28's test outputs, so it runs only when Cell 28 did.
# ============================================================
from blackbox_pipeline.models.rf_router.tuning import ARM_C, ARM_D

if SPEC_EVALUATE_TEST and SPEC_ROUTERS:
    _dC, _dD = SPEC_TEST_OUT[ARM_C][2], SPEC_TEST_OUT[ARM_D][2]
    _y = STAGE2["y_test"].to_numpy()
    print("rows routed differently:", int((_dC != _dD).sum()))
    print("subscribers flagged by only one arm:",
          int((((_dC == "pass2") != (_dD == "pass2")) & (_y == 1)).sum()))
else:
    print("Arm C vs D diagnostic skipped (needs Cell 28's test evaluation).")

## Cell 29 — Stage 3 experiment constants

`XGB_PARAM_SOURCE` selects the XGBoost hyperparameter source:

| Value        | Behaviour                                           |
| ------------ | --------------------------------------------------- |
| `"tune"`     | Run `XGB_TUNE_TRIALS` Optuna trials                 |
| `"explicit"` | Use `XGB_PARAMS_EXPLICIT` directly                  |
| `"artifact"` | Load `best_params` from the newest Stage 3 artifact |

Artifact parameters are valid only for the same split and feature contract.

`_newest()` selects the most recently modified artifact across `XGB_ARTIFACT_DIRS`, excluding `*_latest` copies.

In [ ]:
# ============================================================
# CELL 29: STAGE 3 EXPERIMENT CONSTANTS
# ============================================================
from pathlib import Path

XGB_PARAM_SOURCE = "tune"            # "artifact" | "tune" | "explicit"

XGB_ARTIFACT_DIRS = [BLACKBOX_ROOT / "models" / "xgb"]  # where Cell 30's config saves it

XGB_TUNE_TRIALS = 150                # MIRROR — the EBM's Optuna budget

XGB_PARAMS_EXPLICIT = {              # read only when XGB_PARAM_SOURCE == "explicit"
    "n_estimators": 600,
    "learning_rate": 0.02,
    "max_depth": 4,
    "min_child_weight": 5.0,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "reg_lambda": 1.0,
}


def _newest(dirs, pattern):
    """Newest file by modification time across ALL dirs; skips *_latest copies."""
    found = [p for d in map(Path, dirs) if d.exists()
             for p in d.glob(pattern) if not p.stem.endswith("_latest")]
    return max(found, key=lambda p: p.stat().st_mtime) if found else None


assert XGB_PARAM_SOURCE in ("artifact", "tune", "explicit"), XGB_PARAM_SOURCE
print(f"XGB_PARAM_SOURCE = {XGB_PARAM_SOURCE!r}")


## Cell 30 — Stage 3 feature block

Both arms use the same GLASS feature pipeline:

`drop_leaky_features` → `engineer_ebm_features` → `select_ebm_features`

The black-box arm performs no independent feature engineering.

Assertions verify exact `EBM_FEATURES` order and alignment with `GLOBAL_SPLIT`. Only `y_train` is passed into feature engineering.

In [ ]:
# ============================================================
# CELL 30: STAGE 3 FEATURE BLOCK — per-fold GLASS DAG (same as GLASS 19b)
# ============================================================
from shared.stage3.feature_engineering import EBM_FEATURES, EBMFeaturePipeline
from blackbox_pipeline.models.xgb import XGBStage3Config
from shared.stage3.runner import Stage3Block

XGB_STAGE3_CONFIG = XGBStage3Config(
    expected_features=list(EBM_FEATURES),
    n_trials=XGB_TUNE_TRIALS,
    artifact_dir=str(BLACKBOX_ROOT / "models" / "xgb"),
    verbose=True,
)
assert XGB_STAGE3_CONFIG.n_trials == 150, (
    f"XGB_TUNE_TRIALS={XGB_TUNE_TRIALS}: the EBM arm tunes with 150 trials; "
    "a different budget breaks the comparison."
)

_FP_NOW = split_fingerprint(GLOBAL_SPLIT["X_train"], GLOBAL_SPLIT["X_test"])
STAGE3_BLOCK = Stage3Block.from_global_split(
    GLOBAL_SPLIT, EBMFeaturePipeline, XGB_STAGE3_CONFIG, split_fingerprint=_FP_NOW,
)
assert STAGE3_BLOCK.features == list(EBM_FEATURES)
assert STAGE3_BLOCK.index_train.equals(GLOBAL_SPLIT["X_train"].index)
assert STAGE3_BLOCK.index_test.equals(GLOBAL_SPLIT["X_test"].index)
print(f"block fingerprint: {STAGE3_BLOCK.block_fingerprint}")

## Cell 31 — Stage 3 XGBoost

Trains the black-box counterpart to EBM while matching the split, features, seed, weighting, tuning objective, CV reporting, calibration rule, and threshold grid.

Training-side procedure:

1. Validate features and impute with train medians.
2. Build balanced weights and shared CV folds.
3. Tune with Optuna when required.
4. Produce the 10-fold CV report.
5. Refit on the full training set.
6. Generate OOF predictions for Stage 4.
7. Apply fold-nested isotonic calibration when OOF ECE > 0.05.
8. Select thresholds from OOF predictions.
9. Evaluate on test after all operating choices are frozen.

Recorded departures from GLASS include OOF-based thresholding and calibration, OOF Stage 4 outputs, train-derived imputation, and an XGBoost-specific search space.

`split_fingerprint` is stored for validation in Cell 33.

In [ ]:
# ============================================================
# CELL 31: STAGE 3 — XGBOOST
# ============================================================
import joblib
from blackbox_pipeline.models.xgb import train_xgb_stage
from shared.stage3.protocol import check_param_reuse

_params = _params_source = _params_prov = None
if XGB_PARAM_SOURCE == "explicit":
    # NOTE: no Optuna search on record — the artifact says so (tuning["tuned"] is
    # False) and check_param_reuse will not accept it as a parameter source.
    _params, _params_source = dict(XGB_PARAMS_EXPLICIT), "XGB_PARAMS_EXPLICIT"
elif XGB_PARAM_SOURCE == "artifact":
    _src = _newest(XGB_ARTIFACT_DIRS, "xgb_stage3_*.joblib")
    if _src is None:
        raise FileNotFoundError(
            f"XGB_PARAM_SOURCE='artifact' but no xgb_stage3_*.joblib in "
            f"{[str(d) for d in XGB_ARTIFACT_DIRS]}. Use 'tune' first."
        )
    _reuse = check_param_reuse(
        joblib.load(_src), XGB_STAGE3_CONFIG, _FP_NOW,
        STAGE3_BLOCK.block_fingerprint, list(EBM_FEATURES),
    )   # checks split, block, features, n_trials, search space, scope, seeds
    _params, _params_source, _params_prov = (
        _reuse["params"], f"{_src.name}: {_reuse['source']}", _reuse["provenance"])
    print(f"params ← {_params_source}")

XGB_STAGE3, XGB_STAGE3_PATH = train_xgb_stage(
    block=STAGE3_BLOCK,
    config=XGB_STAGE3_CONFIG,
    params=_params,
    params_source=_params_source,
    params_provenance=_params_prov,
)

## Cell 32 — Stage 4 hand-off

| Accessor              | Source                | Use              |
| --------------------- | --------------------- | ---------------- |
| `stage4_frame()`      | OOF train predictions | Train Stage 4    |
| `stage4_test_frame()` | Refit model on test   | Evaluate Stage 4 |

Both expose `proba`, `proba_calibrated`, `decision`, `confidence`, `margin`, `state`, and `fold_id`.

Train outputs are leakage-safe: probabilities are OOF, calibration is fold-nested, and decisions use per-fold thresholds chosen without that fold's labels.

Do not train Stage 4 on `refit["train_proba"]`; it is in-sample and diagnostic only.

Threshold spread measures stability. OOF F2 using a threshold selected on the same rows is mildly optimistic.

In [ ]:
# ============================================================
# CELL 32: STAGE 4 HAND-OFF — OOF vs REFIT
# ============================================================
from sklearn.metrics import roc_auc_score

STAGE3_XGB_TRAIN = XGB_STAGE3.stage4_frame()             # OOF features — train Stage 4
STAGE3_XGB_TEST = XGB_STAGE3.stage4_test_frame()         # refit features — evaluate Stage 4
STAGE3_XGB_TEST_FOLDMEAN = XGB_STAGE3.stage4_test_frame(source="fold_mean")
STAGE3_XGB_TRAIN_META = XGB_STAGE3.stage4_metadata()     # NOT features

assert STAGE3_XGB_TRAIN.index.equals(GLOBAL_SPLIT["X_train"].index)
assert STAGE3_XGB_TEST.index.equals(GLOBAL_SPLIT["X_test"].index)
assert not any(c.endswith(("fold_id", "state", "threshold")) for c in STAGE3_XGB_TRAIN.columns)

print(STAGE3_XGB_TRAIN.head())
print(f"\ntrain side {STAGE3_XGB_TRAIN.shape} | test side {STAGE3_XGB_TEST.shape}")

_y = GLOBAL_SPLIT["y_train"]
_auc_oof = roc_auc_score(_y, XGB_STAGE3.oof["proba"])
_auc_ins = roc_auc_score(_y, XGB_STAGE3.refit["train_proba"])
print(f"\ntrain AUC  OOF {_auc_oof:.4f}  |  in-sample {_auc_ins:.4f}  "
      f"|  optimism {_auc_ins - _auc_oof:+.4f}   (features fitted "
      f"{XGB_STAGE3.feature_fit_scope})")

_nest = XGB_STAGE3.threshold["oof_nested"]
_m_sel = XGB_STAGE3.metrics["oof_at_operating_threshold"]
_m_nst = XGB_STAGE3.metrics["oof_at_nested_thresholds"]
print(f"\nthreshold  global {XGB_STAGE3.threshold['oof']['threshold']:.4f}  |  "
      f"per-fold {_nest['by_fold']}  (sd {_nest['std']:.4f})")
print(f"OOF F2     in-selection {_m_sel['f2']:.4f}  |  fold-nested {_m_nst['f2']:.4f}")

_cal = XGB_STAGE3.calibration
print(f"calibration {'applied' if _cal['applied'] else 'skipped'}  |  "
      f"OOF ECE {_cal['ece_before']:.4f} → {_cal['ece_after']:.4f}  |  "
      f"test ECE {_cal['test_ece_raw']:.4f} → {_cal['test_ece_calibrated']:.4f}  "
      f"({_cal['n_zero_after']} OOF rows at p=0)")
_rf = XGB_STAGE3.metrics["test_at_operating_threshold"]
_fm = XGB_STAGE3.metrics["test_fold_mean_at_operating_threshold"]
print(f"test F2    refit {_rf['f2']:.4f}  |  fold-mean {_fm['f2']:.4f}")

## Cell 33 — EBM vs XGBoost

Finds the GLASS Stage 3 EBM that Cell 34 compares with XGBoost on the same test rows and Cell 37 hands to the crossover.

- Reads the EBM this run pinned (`$BANK_UPSTREAM_PINS`, role `stage3.ebm`): the file GLASS Cell 18 wrote, by path and SHA-256, never the newest file (PR 39, F9).
- Requires the artifact to match this split (either fingerprint scheme), the shared block fingerprint and the feature list, and its training and test rows to be `GLOBAL_SPLIT`'s.
- Asserts the GLASS EBM identity (`model_family='ebm'`, `arm='glass'`) before anything uses it.

Both arms run the shared Stage 3 protocol, so both operating thresholds were selected on out-of-fold training predictions; each artifact keeps its test-fitted threshold only as a labelled reference.

In [ ]:
# ============================================================
# CELL 33: STAGE 3 — EBM vs XGBOOST
# ============================================================
import pandas as pd
from shared.pinned_artifacts import pinned_path
from shared.stage_io import split_fingerprint as _fp_stage_io
from shared.stage3.artifact import Stage3ArtifactStore

assert XGB_STAGE3.split_fingerprint == _FP_NOW, "XGB_STAGE3 is from another split — re-run Cell 31."
assert pd.Index(XGB_STAGE3.index_test).equals(GLOBAL_SPLIT["X_test"].index)

# The GLASS EBM this run pinned (PR 39, F9): the file GLASS Cell 18 wrote, by path and
# SHA-256 from $BANK_UPSTREAM_PINS, never the newest file in models/ebm.
_ebm_path = pinned_path("stage3.ebm")
EBM_STAGE3 = Stage3ArtifactStore.load(str(_ebm_path))
print(f"GLASS EBM ← {_ebm_path}")

# The GLASS notebook stamps Stage 3 with the stage_io fingerprint; this notebook
# uses the router format. Same split, two formats — either matches.
_FP_CANDIDATES = [
    _FP_NOW,
    _fp_stage_io(GLOBAL_SPLIT["X_train"].index, GLOBAL_SPLIT["X_test"].index,
                 GLOBAL_SPLIT["y_train"].astype(int), GLOBAL_SPLIT["y_test"].astype(int)),
]
assert EBM_STAGE3.split_fingerprint in _FP_CANDIDATES, f"{_ebm_path} is from another split"
assert EBM_STAGE3.block_fingerprint == STAGE3_BLOCK.block_fingerprint, (
    f"{_ebm_path} was built on another Stage 3 block")
assert list(EBM_STAGE3.features) == list(EBM_FEATURES), f"{_ebm_path} has other features"

_ebm_identity = (EBM_STAGE3.model_family, (EBM_STAGE3.config or {}).get("arm"))
assert _ebm_identity == ("ebm", "glass"), (
    f"{_ebm_path} is not a GLASS EBM artifact (model_family, arm = {_ebm_identity}) — "
    "re-run GLASS Cells 17-18")
assert pd.Index(EBM_STAGE3.index_train).equals(GLOBAL_SPLIT["X_train"].index), "EBM train rows differ"
assert pd.Index(EBM_STAGE3.index_test).equals(GLOBAL_SPLIT["X_test"].index), "EBM test rows differ"

## Cell 34 — Statistical Comparison

| Output                   | Purpose                                                        |
| ------------------------ | -------------------------------------------------------------- |
| Paired bootstrap CIs     | 95% CI on each XGBoost − EBM difference; real only if it excludes 0 |
| PR-AUC                   | Top-of-list ranking quality at an 11% positive rate            |
| Test-fitted thresholds   | Both models at a `y_test`-fitted threshold — the protocol-matched operating view (reference only) |
| Calibrated test metrics  | Whether XGBoost's OOF-fitted isotonic calibration holds on test |
| Spearman correlation     | Prediction overlap; lower means more complementary signal for Stage 4 |

The bootstrap uses 2,000 resamples (seed 42). Each resample scores both models on the same rows, so the interval covers the difference directly. Metrics use raw probabilities. A negative Brier Δ means XGBoost is better.

In [ ]:
# ============================================================
# CELL 34: STAGE 3 — STATISTICAL COMPARISON
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

_y = np.asarray(GLOBAL_SPLIT["y_test"]).astype(int)
_pe = np.asarray(EBM_STAGE3.refit["test_proba"], dtype=float)                 # ranking
_px = np.asarray(XGB_STAGE3.refit["test_proba"], dtype=float)
_pe_c = np.asarray(EBM_STAGE3.refit["test_proba_calibrated"], dtype=float)    # probability quality
_px_c = np.asarray(XGB_STAGE3.refit["test_proba_calibrated"], dtype=float)
BUDGETS, N_BOOT, SEED = (0.05, 0.10, 0.20, 0.30), 2000, 42


def _recall_at(p, y, rate):
    return ((p >= np.quantile(p, 1 - rate)) & (y == 1)).sum() / y.sum()


def _per_arm(p, pc, y):
    out = {"ROC-AUC": roc_auc_score(y, p),
           "PR-AUC": average_precision_score(y, p),
           "Brier, calibrated (lower=better)": brier_score_loss(y, pc)}
    out.update({f"recall @ {r:.0%} budget": _recall_at(p, y, r) for r in BUDGETS})
    return out


# Paired bootstrap: the same resampled rows score both arms.
_point_e, _point_x = _per_arm(_pe, _pe_c, _y), _per_arm(_px, _px_c, _y)
_rng = np.random.default_rng(SEED)
_deltas = {k: [] for k in _point_e}
for _ in range(N_BOOT):
    _idx = _rng.integers(0, len(_y), len(_y))
    _yb = _y[_idx]
    if _yb.min() == _yb.max():
        continue
    _e = _per_arm(_pe[_idx], _pe_c[_idx], _yb)
    _x = _per_arm(_px[_idx], _px_c[_idx], _yb)
    for k in _deltas:
        _deltas[k].append(_x[k] - _e[k])

_rows = []
for k in _point_e:
    _d = np.asarray(_deltas[k])
    _lo, _hi = np.percentile(_d, [2.5, 97.5])
    _better = (_d < 0).mean() if k.startswith("Brier") else (_d > 0).mean()
    _rows.append({"metric": k, "EBM": _point_e[k], "XGBoost": _point_x[k],
                  "Δ (XGB−EBM)": _point_x[k] - _point_e[k],
                  "95% CI low": _lo, "95% CI high": _hi,
                  "P(XGB better)": _better, "CI excludes 0": not (_lo <= 0 <= _hi)})
STAGE3_STATS = pd.DataFrame(_rows)

# Both arms at their OOF-selected operating points — like-for-like now.
_COLS = {"threshold": "threshold", "f2": "F2", "recall": "recall",
         "precision": "precision", "roc_auc": "ROC-AUC", "brier": "Brier", "ece": "ECE"}
STAGE3_OPERATING = pd.DataFrame({
    f"{name} {view}": {v: art.metrics[key][k] for k, v in _COLS.items()}
    for name, art in (("EBM", EBM_STAGE3), ("XGBoost", XGB_STAGE3))
    for view, key in (("raw @ OOF thr", "test_at_operating_threshold"),
                      ("calibrated @ OOF thr", "test_calibrated_at_operating_threshold"),
                      ("oracle @ test thr (REFERENCE)", "test_at_oracle_threshold_REFERENCE_ONLY"))
}).T

_rho = spearmanr(_pe, _px).correlation

print(f"paired bootstrap, {N_BOOT} resamples, seed {SEED}, n={len(_y):,}\n")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(STAGE3_STATS.to_string(index=False))
    print("\noperating points (thresholds selected on OOF train predictions; oracle = reference only)")
    print(STAGE3_OPERATING.to_string())
print(f"\nprediction correlation (Spearman, test): {_rho:.4f}")

## Cell 35 — Stage 4 Hand-off Summary

Runs only after validation passes and summarizes the signals entering Stage 4.

| Output           | Contents                                          |
| ---------------- | ------------------------------------------------- |
| `STAGE4_SCORERS` | Stage 1/3 thresholds, OOF vs test AUC, flag rates |
| `STAGE4_ROUTER`  | Stage 2 route shares and abstain/base rates       |
| Figure           | OOF vs test scoring and routing behavior          |
| Crosstab         | Stage 2 test decision encoding                    |

**Check for:**

* similar OOF and test behavior;
* higher positive base rate among Stage 2 abstains than overall.

Large OOF/test gaps may indicate unstable generalization into Stage 4.

In [ ]:
# ============================================================
# CELL 35: STAGE 4 — META-XGB INPUTS
# ============================================================
from blackbox_pipeline.models.meta_xgb import MetaXGBConfig, train_meta_xgb_stage
from shared.pinned_artifacts import pinned_path

# Every Stage 4 input by path (PR 39, F9): the RF Router from Cell 18 (the accepted PR 38
# run's immutable version, pinned) and the GLASS Arbiter GLASS Cell 19 wrote, by path and
# SHA-256 from $BANK_UPSTREAM_PINS; never the newest file or a latest alias.
GLASS_ARBITER_PATH = pinned_path("stage4.glass_arbiter")
print(f"RF Router     ← {RF_ROUTER_PATH}")
print(f"GLASS Arbiter ← {GLASS_ARBITER_PATH}")

META_XGB_CONFIG = MetaXGBConfig(artifact_dir=str(BLACKBOX_ROOT / "models" / "meta_xgb"))

In [ ]:
# ============================================================
# CELL 36: STAGE 4 — META-XGB (train-only config → test once → compare)
# ============================================================
META_XGB, META_XGB_PATH = train_meta_xgb_stage(
    GLOBAL_SPLIT,
    mlp_path=MLP_S1_PATH,
    rf_path=RF_ROUTER_PATH,
    xgb_path=XGB_STAGE3_PATH,
    config=META_XGB_CONFIG,
    glass_arbiter_path=GLASS_ARBITER_PATH,
)

In [ ]:
# ============================================================
# CELL 37: STAGE 4 — 2×2 ARBITER CROSSOVER
# ============================================================
# Each arbiter on each arm's saved Stage 1–3 outputs. The crossover is declared
# (upstream=...): every artifact must then belong to that arm, and the artifact
# records the combination and exactly which outputs filled the arbiter's slots
# (stage4_inputs["upstream"]; PR 36 F29). The GLASS Arbiter reads the RF
# Router's band blend as its router evidence (band rate vs base rate, F25).
from glass_pipeline.glass_arbiter.meta_stage import train_glass_arbiter
from blackbox_pipeline.models.meta_xgb import MetaXGBConfig, train_meta_xgb_stage, compare_with_glass_arbiter

_XO_DIR = BLACKBOX_ROOT / "models" / "stage4_crossover"

# GLASS Arbiter on black-box upstream (MLP, RF Router, XGBoost)
_, XO_ARB_ON_BB_PATH = train_glass_arbiter(
    GLOBAL_SPLIT, lr_path=MLP_S1_PATH, glass_path=RF_ROUTER_PATH, ebm_path=XGB_STAGE3_PATH,
    target_recall=0.70, artifact_dir=str(_XO_DIR / "glass_arbiter_on_blackbox"),
    upstream="blackbox",
)

# Meta-XGB on GLASS upstream (LR, GLASS Router, EBM)
XO_XGB_ON_GLASS, XO_XGB_ON_GLASS_PATH = train_meta_xgb_stage(
    GLOBAL_SPLIT, mlp_path=GLASS_S1_PATH, rf_path=GLASS_BUNDLE_PATH, xgb_path=_ebm_path,
    config=MetaXGBConfig(artifact_dir=str(_XO_DIR / "meta_xgb_on_glass")),
    upstream="glass",
)

In [ ]:
# ============================================================
# CELL 38: STAGE 4 — CROSSOVER RESULTS (arbiter effect, upstream held fixed)
# ============================================================
print("=== GLASS upstream: GLASS Arbiter vs Meta-XGB ===")
XO_GLASS_COLUMN = compare_with_glass_arbiter(GLOBAL_SPLIT, GLASS_ARBITER_PATH, XO_XGB_ON_GLASS,
                                             require_same_upstream=True)

print("\n=== Black-box upstream: GLASS Arbiter vs Meta-XGB ===")
XO_BB_COLUMN = compare_with_glass_arbiter(GLOBAL_SPLIT, XO_ARB_ON_BB_PATH, META_XGB,
                                          require_same_upstream=True)